# Comparativo Institucional de Relatórios Mensais (por Ano)

Este notebook define um ano, procura na subpasta `.../dados_viagens_{ano}/Relatorios_Mensais/` por todos os CSVs de relatórios (ex: UFPB, UFCG) e gera um dashboard comparativo **institucional** para aquele ano.

## 1. Configuração do Ano e Imports

In [30]:
import pandas as pd
import numpy as np
import altair as alt
import os
import glob
import re

# --- Configurações ---
ano = 2025 # Ano que você quer comparar

# --- Caminho para a PASTA DE ENTRADA (Relatórios Mensais) ---
pasta_dados_ano = f'dadosViagens/dados_viagens{ano}'
PASTA_RELATORIOS_MENSAIS = os.path.join(pasta_dados_ano, 'Relatorios_Mensais')

if not os.path.exists(PASTA_RELATORIOS_MENSAIS):
    print(f"⚠️ Aviso: Pasta de relatórios não encontrada em '{PASTA_RELATORIOS_MENSAIS}'. Criando...")
    os.makedirs(PASTA_RELATORIOS_MENSAIS)

print(f"CONFIGURAÇÃO: Comparando instituições para o ano {ano}")
print(f"-> Lendo relatórios de: {PASTA_RELATORIOS_MENSAIS}")

CONFIGURAÇÃO: Comparando instituições para o ano 2025
-> Lendo relatórios de: dadosViagens/dados_viagens2025\Relatorios_Mensais


## 2. Carregar Dados Comparativos

In [31]:
print(f"🔄 Carregando todos os relatórios da pasta: '{PASTA_RELATORIOS_MENSAIS}'...")

# Procura todos os arquivos do ANO CORRENTE na subpasta
search_pattern = os.path.join(PASTA_RELATORIOS_MENSAIS, f"relatorio_mensal_*_aereo_{ano}.csv")
all_files = glob.glob(search_pattern)
all_files.sort()

all_data_list = []

if not all_files:
    print(f"❌ Nenhum arquivo encontrado com o padrão: {search_pattern}")
    print(f"   (Execute o notebook 'gerar_relatorio_mensal.ipynb' para pelo menos uma instituição)")
else:
    print(f"✅ Encontrados {len(all_files)} arquivos para comparação do ano {ano}:")
    for f in all_files:
        print(f"   - {os.path.basename(f)}")
        try:
            # Extrai a INSTITUIÇÃO do nome do arquivo
            org_match = re.search(f'relatorio_mensal_(.*?)_aereo_{ano}\.csv$', os.path.basename(f))
            if not org_match:
                print(f"     ⚠️ Não foi possível extrair a instituição do nome do arquivo, pulando.")
                continue
                
            instituicao_arquivo = org_match.group(1)
            
            df_report = pd.read_csv(f)
            df_report['Instituicao'] = instituicao_arquivo
            all_data_list.append(df_report)
            
        except Exception as e_load_comp:
            print(f"     ❌ Erro ao carregar o arquivo {f}: {e_load_comp}")

if all_data_list:
    df_comparativo = pd.concat(all_data_list, ignore_index=True)
    print("\n✅ Todos os relatórios combinados em 'df_comparativo'.")
    print("\n--- Instituições Carregadas ---")
    print(df_comparativo['Instituicao'].value_counts().sort_index())
else:
    print("\n⚠️ Nenhum dado comparativo foi carregado.")
    df_comparativo = pd.DataFrame() # Cria vazio para evitar erros

🔄 Carregando todos os relatórios da pasta: 'dadosViagens/dados_viagens2025\Relatorios_Mensais'...
✅ Encontrados 2 arquivos para comparação do ano 2025:
   - relatorio_mensal_UFCG_aereo_2025.csv
   - relatorio_mensal_UFPB_aereo_2025.csv

✅ Todos os relatórios combinados em 'df_comparativo'.

--- Instituições Carregadas ---
Instituicao
UFCG    12
UFPB    12
Name: count, dtype: int64


## 3. Visualização Comparativa (Institucional por Ano)

In [32]:
instituicao_arquivo = "_e_".join(sorted(df_comparativo['Instituicao'].unique()))
instituicao_arquivo

'UFCG_e_UFPB'

In [33]:
if 'df_comparativo' in locals() and not df_comparativo.empty:
    print("🔄 Criando dashboard comparativo institucional com rótulos de dados...")
    
    # Gráfico base comparativo
    base_comp = alt.Chart(df_comparativo).encode(
        x=alt.X('Mes_Num:O', axis=alt.Axis(title='Mês', labels=True, ticks=True)),
        color=alt.Color('Instituicao:N', title='Instituição'),
        tooltip=['Instituicao', 'Mes_Ano', 
                 alt.Tooltip('Total_Emissoes_KgCO2eq', title='Emissões ($KgCO_2eq$)', format=',.0f'),
                 alt.Tooltip('Total_Distancia_Km', title='Distância (km)', format=',.0f'),
                 alt.Tooltip('Total_Passagens', title='Passagens (R$)', format=',.2f')
                ]
    ).properties(
        width=700
    )
    
    # --- Gráfico 1: Emissões (Linhas + Texto) ---
    line_comp_emissoes = base_comp.mark_line(point=True).encode(
        y=alt.Y('Total_Emissoes_KgCO2eq', title='Total Emissões ($KgCO_2eq$)')
    )
    text_comp_emissoes = base_comp.mark_text(dy=-10, color='black').encode(
        y=alt.Y('Total_Emissoes_KgCO2eq'),
        text=alt.Text('Total_Emissoes_KgCO2eq', format=',.0f')
    )
    chart_comp_emissoes = (line_comp_emissoes + text_comp_emissoes).properties(
        title='Comparativo Institucional de Emissões'
    )
    
    # --- Gráfico 2: Distância (Linhas + Texto) ---
    line_comp_distancia = base_comp.mark_line(point=True).encode(
        y=alt.Y('Total_Distancia_Km', title='Total Distância (km)')
    )
    text_comp_distancia = base_comp.mark_text(dy=-10, color='black').encode(
        y=alt.Y('Total_Distancia_Km'),
        text=alt.Text('Total_Distancia_Km', format=',.0f')
    )
    chart_comp_distancia = (line_comp_distancia + text_comp_distancia).properties(
        title='Comparativo Institucional de Distância'
    )

    # --- Gráfico 3: Passagens (Linhas + Texto) ---
    line_comp_passagens = base_comp.mark_line(point=True).encode(
        y=alt.Y('Total_Passagens', title='Total Passagens (R$)')
    )
    text_comp_passagens = base_comp.mark_text(dy=-10, color='black').encode(
        y=alt.Y('Total_Passagens'),
        text=alt.Text('Total_Passagens', format=',.0f')
    )
    chart_comp_passagens = (line_comp_passagens + text_comp_passagens).properties(
        title='Comparativo Institucional de Custo de Passagens'
    )

    dashboard_comparativo = alt.vconcat(
        chart_comp_emissoes,
        chart_comp_distancia,
        chart_comp_passagens
    ).properties(
        title=f"Comparativo Institucional de Tendências Mensais - Ano {ano}"
    ).interactive()
    
    # --- SALVAR O DASHBOARD COMO PDF ---
    try:
        # *** ALTERAÇÃO AQUI: Mudando a extensão para .pdf ***
        arquivo_dashboard_comp = os.path.join(PASTA_RELATORIOS_MENSAIS, f'dashboard_comparativo_institucional_{ano}_{instituicao_arquivo}.pdf')
        
        # O método .save() usará 'vl-convert-python' automaticamente
        # se a extensão .pdf for detectada e o pacote estiver instalado.
        dashboard_comparativo.save(arquivo_dashboard_comp)
        
        print(f"\n✅ Dashboard COMPARATIVO salvo como PDF em: '{arquivo_dashboard_comp}'")
    except Exception as e_save_comp:
        print(f"❌ Erro ao salvar dashboard comparativo: {e_save_comp}")
        print("   -> Certifique-se de que você instalou 'vl-convert-python' (pip install vl-convert-python)")

else:
    print("⚠️ 'df_comparativo' vazio. Gráfico comparativo pulado.")


🔄 Criando dashboard comparativo institucional com rótulos de dados...

✅ Dashboard COMPARATIVO salvo como PDF em: 'dadosViagens/dados_viagens2025\Relatorios_Mensais\dashboard_comparativo_institucional_2025_UFCG_e_UFPB.pdf'


In [34]:
dashboard_comparativo # Mostra o gráfico comparativo final

alt.VConcatChart(...)